In [1]:
import pickle
from pathlib import Path

In [10]:
data_dir = Path('/u/sdickman/DRAKES/drakes_protein/fmif/align_oracle_data')
target_pkl = "full_traj_train_pretrained.pkl"
target_path = data_dir / target_pkl
print(target_path)

with open(data_dir / target_pkl, 'rb') as f:
    data = pickle.load(f)
    print(len(data))

for d in data:
    if d['t'] == 1:
        print(d)


/u/sdickman/DRAKES/drakes_protein/fmif/align_oracle_data/full_traj_train_pretrained.pkl
1654500
{'protein_name': 'EA|run2_0325_0005', 't': 1, 'sequence': '------------T----------------------------------', 'final_align_reward': -0.34924355149269104, 'final_eval_reward': -1.1934212446212769}
{'protein_name': 'EA|run2_0325_0005', 't': 1, 'sequence': '------------T----------------------------------', 'final_align_reward': -0.34924355149269104, 'final_eval_reward': -1.1934212446212769}
{'protein_name': 'EA|run2_0325_0005', 't': 1, 'sequence': '------------T----------------------------------', 'final_align_reward': -0.34924355149269104, 'final_eval_reward': -1.1934212446212769}
{'protein_name': 'EA|run2_0325_0005', 't': 1, 'sequence': '------------T----------------------------------', 'final_align_reward': -0.34924355149269104, 'final_eval_reward': -1.1934212446212769}
{'protein_name': 'EA|run2_0325_0005', 't': 1, 'sequence': '------------T----------------------------------', 'final_align_re

In [11]:
# For each full_traj_*_pretrained.pkl, keep only the FIRST trajectory per protein.
# Each record is one denoising step {'protein_name', 't', 'sequence', ...}. A single
# trajectory is a contiguous block of steps whose index `t` increases from 1; a new
# trajectory for the same protein begins whenever `t` resets back to 1.

def keep_first_trajectory(records):
    started = set()  # proteins we've begun collecting
    done = set()     # proteins whose first trajectory is complete
    out = []
    for rec in records:
        name = rec['protein_name']
        if name in done:
            continue
        if rec['t'] == 1 and name in started:
            # start of a second trajectory for this protein -> first one is finished
            done.add(name)
            continue
        started.add(name)
        out.append(rec)
    return out


for src in sorted(data_dir.glob('full_traj_*_pretrained.pkl')):
    with open(src, 'rb') as f:
        records = pickle.load(f)
    first = keep_first_trajectory(records)
    dst = src.with_name(f'{src.stem}_first_traj.pkl')
    with open(dst, 'wb') as f:
        pickle.dump(first, f)
    n_prot = len({r['protein_name'] for r in first})
    print(f'{src.name}: {len(records)} -> {len(first)} records, {n_prot} proteins -> {dst.name}')

full_traj_test_pretrained.pkl: 58600 -> 586 records, 12 proteins -> full_traj_test_pretrained_first_traj.pkl
full_traj_train_pretrained.pkl: 1654500 -> 16545 records, 339 proteins -> full_traj_train_pretrained_first_traj.pkl
full_traj_validation_pretrained.pkl: 49000 -> 490 records, 10 proteins -> full_traj_validation_pretrained_first_traj.pkl


In [12]:
# Verify each new file has exactly one trajectory per protein
for dst in sorted(data_dir.glob('full_traj_*_pretrained_first_traj.pkl')):
    with open(dst, 'rb') as f:
        records = pickle.load(f)
    # count trajectories per protein by detecting t==1 resets
    trajs_per_protein = {}
    for rec in records:
        name = rec['protein_name']
        if rec['t'] == 1:
            trajs_per_protein[name] = trajs_per_protein.get(name, 0) + 1
    max_trajs = max(trajs_per_protein.values())
    print(f'{dst.name}: {len(records)} records, {len(trajs_per_protein)} proteins, '
          f'max trajectories/protein = {max_trajs}')
    assert max_trajs == 1, f'{dst.name} still has multiple trajectories for some protein!'

full_traj_test_pretrained_first_traj.pkl: 586 records, 12 proteins, max trajectories/protein = 1
full_traj_train_pretrained_first_traj.pkl: 16545 records, 339 proteins, max trajectories/protein = 1
full_traj_validation_pretrained_first_traj.pkl: 490 records, 10 proteins, max trajectories/protein = 1
